Session 2026-06-24 - MTL built from scratch through to stage 1 fit

State
- mtl_da_long: 19,902,240 rows (6504 DA x 4 band x 765 day). cols da_idx, age_band, pop, annual_rate, date, lambda0. lambda0 = annual_rate*pop/1000/365.
- mtl_sim: same 19.9M rows + n_deaths, log_rr, lambda, date_idx. Clean: 0 NA, total 142,934 deaths, range 0-36/DA-band-day, 99.3% zero
- mmt_da: length 6504, per-DA 80th-pct temp, range 20.5-22.5C
- mtl_7584: age_75_84 slice, setorder(da_idx, date), temp attached, cb matrix-column attached, da_idx renamed DA_id
- cb_mtl: 4,975,640 x 25 cross-basis (regenerable, ~1.2GB, not stored).
- fit_7584: waiting...

What got done
restored env (clobber made clean qaic win over stale) -> found MTL was never simulated -> built da_long by hand -> simulated full MTL clean -> assembled fitting frame -> fit Stage 1

Learned
- lambda0 = annual_rate*pop/1000/365, recovered by matching toronto rows
- cma_age_data arg name lies: fit_stage 1 uses it as the long fitting frame, not the pop table that shares the name
- temp onto long rows = temp_mat[cbind(da_idx, date_idx)] - matrix lookup, no join, can't misalign.
- attach a cb with base $<-, not := (:= flattens the 25-col matrix into one vector, errors 25x)
- row order is da_idx>date>band; impose setorder(da_idx,date) after band-subset before building cb.

Pick up from:
read fit_7584.

Open:
- 0-64 band is stress case (low deaths/param) - drop from pilot stage 2 if it fights dont fight it
- MTL 75-84 fit ran 18+ min, may need a time budget or DRAC for full bronzing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

---



In [ ]:
list.files("/content/drive/MyDrive/thesis/dlnm-pilot")

[1] "DLNM-Play.ipynb"                "montreal_da_shp.zip"           
 [3] "montreal_daymet_2015_2019.csv"  "r_library.tar.gz"              
 [5] "saves_2026-05-26.tar.gz"        "saves_eod_2026-06-10.tar.gz"   
 [7] "saves_eod_2026-06-17.tar.gz"    "saves_mtlvan_clean.tar.gz"     
 [9] "saves_pilot_2026-05-27.tar.gz"  "saves_pilot_2026-06-03.tar.gz" 
[11] "toronto_da_shp.zip"             "toronto_da.geojson"            
[13] "toronto_daymet_2015_2019.csv"   "vancouver_da_shp.zip"          
[15] "vancouver_daymet_2015_2019.csv"

'DLNM-Play.ipynb''montreal_da_shp.zip''montreal_daymet_2015_2019.csv''r_library.tar.gz''saves_2026-05-26.tar.gz''saves_eod_2026-06-10.tar.gz''saves_eod_2026-06-17.tar.gz''saves_mtlvan_clean.tar.gz''saves_pilot_2026-05-27.tar.gz''saves_pilot_2026-06-03.tar.gz''toronto_da_shp.zip''toronto_da.geojson''toronto_daymet_2015_2019.csv''vancouver_da_shp.zip''vancouver_daymet_2015_2019.csv'

---


In [ ]:
setwd("/content/drive/MyDrive/thesis/dlnm-pilot")
untar("r_library.tar.gz", exdir = "/content/r_library")
.libPaths(c("/content/r_library", .libPaths()))
.libPaths()

[1] "/content/r_library"            "/usr/local/lib/R/site-library"
[3] "/usr/lib/R/site-library"       "/usr/lib/R/library"

'/content/r_library''/usr/local/lib/R/site-library''/usr/lib/R/site-library''/usr/lib/R/library'

--


In [ ]:
list.files("/content/r_library")[1:15]

[1] "site-library" NA             NA             NA             NA            
 [6] NA             NA             NA             NA             NA            
[11] NA             NA             NA             NA             NA

'site-library'NANANANANANANANANANANANANANA

In [ ]:
.libPaths(c("/content/r_library/site-library", .libPaths()))
list.files("/content/r_library/site-library")[1:15]

[1] "abind"        "arrow"        "AsioHeaders"  "assertthat"   "BH"          
 [6] "bigD"         "bitops"       "ca"           "cancensus"    "chromote"    
[11] "classInt"     "colorspace"   "colourvalues" "cols4all"     "covr"

'abind''arrow''AsioHeaders''assertthat''BH''bigD''bitops''ca''cancensus''chromote''classInt''colorspace''colourvalues''cols4all''covr'

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(dlnm)
  library(splines)
  library(gnm)
  library(mixmeta)
  library(sf)
})
cat("dlnm:", as.character(packageVersion("dlnm")),
    "| mixmeta:", as.character(packageVersion("mixmeta")),
    "| gnm:", as.character(packageVersion("gnm")),
    "| data.table:", as.character(packageVersion("data.table")), "\n")

dlnm: 2.4.10 | mixmeta: 1.2.2 | gnm: 1.1.5 | data.table: 1.18.4 


dlnm: 2.4.10 | mixmeta: 1.2.2 | gnm: 1.1.5 | data.table: 1.18.4

[7]
7s


In [ ]:
untar("saves_pilot_2026-06-03.tar.gz", exdir = "/content/pilot_0603")
list.files("/content/pilot_0603", recursive = TRUE)

[1] "saves/da_age_wide.rds"        "saves/mmt_da.rds"            
 [3] "saves/pilot_session.RData"    "saves/reduce_sliver_red.rds" 
 [5] "saves/sim_with_deaths.rds"    "saves/stage1_sliver_res.rds" 
 [7] "saves/temp_mat.rds"           "saves/toronto_da_full.rds"   
 [9] "saves/toronto_dauids.rds"     "saves/truth_factors_trim.rds"
[11] "saves/Z_trim.rds"

'saves/da_age_wide.rds''saves/mmt_da.rds''saves/pilot_session.RData''saves/reduce_sliver_red.rds''saves/sim_with_deaths.rds''saves/stage1_sliver_res.rds''saves/temp_mat.rds''saves/toronto_da_full.rds''saves/toronto_dauids.rds''saves/truth_factors_trim.rds''saves/Z_trim.rds'

In [ ]:
before <- ls()
load("/content/pilot_0603/saves/pilot_session.RData")
injected <- setdiff(ls(), c(before, "before"))
injected

[1] "%||%"                "annual_rates"        "base_log_rr"        
 [4] "build_crossbasis"    "cdn_2011_std"        "cma_list"           
 [7] "compute_da_mmt"      "da_age"              "da_age_wide"        
[10] "da_long"             "daymet_csv"          "F_mat"              
[13] "fit_da_pca"          "fit_stage1"          "fit_stage2"         
[16] "L"                   "lag_weights"         "lib_dest"           
[19] "lib_tarball"         "make_choropleth"     "make_strata_A"      
[22] "make_strata_B"       "monte_carlo_ci"      "n_da"               
[25] "n_pkgs"              "old_wd"              "out_dir"            
[28] "out_geojson"         "pop_long"            "predict_da_theta"   
[31] "qaic"                "reduce_fit"          "saves_tarball"      
[34] "simulate_counts"     "snapshot_dir"        "snapshot_file"      
[37] "standardize_da_rate" "study_dates"         "toronto_da_full"    
[40] "toronto_dauids"      "true_loadings"       "truth_factors"      
[43] "warm_months"         "year_end"            "year_start"         
[46] "Z"                   "zip_path"

'%||%''annual_rates''base_log_rr''build_crossbasis''cdn_2011_std''cma_list''compute_da_mmt''da_age''da_age_wide''da_long''daymet_csv''F_mat''fit_da_pca''fit_stage1''fit_stage2''L''lag_weights''lib_dest''lib_tarball''make_choropleth''make_strata_A''make_strata_B''monte_carlo_ci''n_da''n_pkgs''old_wd''out_dir''out_geojson''pop_long''predict_da_theta''qaic''reduce_fit''saves_tarball''simulate_counts''snapshot_dir''snapshot_file''standardize_da_rate''study_dates''toronto_da_full''toronto_dauids''true_loadings''truth_factors''warm_months''year_end''year_start''Z''zip_path'

In [ ]:
untar("saves_eod_2026-06-17.tar.gz", exdir = "/content/eod_0617")
list.files("/content/eod_0617", recursive = TRUE)

[1] "saves_eod/cma_age_data_mtlvan.rds" "saves_eod/fn_fit_stage1.rds"      
[3] "saves_eod/fn_qaic.rds"             "saves_eod/fn_reduce_fit.rds"      
[5] "saves_eod/mtl_substrate.rds"       "saves_eod/van_substrate.rds"

'saves_eod/cma_age_data_mtlvan.rds''saves_eod/fn_fit_stage1.rds''saves_eod/fn_qaic.rds''saves_eod/fn_reduce_fit.rds''saves_eod/mtl_substrate.rds''saves_eod/van_substrate.rds'

restoring three fixed fx over stale ones the old Rdata just dumped in. body(qaic) prints fx code so we eyeball which version won

In [ ]:
qaic        <- readRDS("/content/eod_0617/saves_eod/fn_qaic.rds")
reduce_fit  <- readRDS("/content/eod_0617/saves_eod/fn_reduce_fit.rds")
fit_stage1  <- readRDS("/content/eod_0617/saves_eod/fn_fit_stage1.rds")
body(qaic)

{
    phi <- summary(fit)$dispersion
    deviance(fit)/phi + 2 * length(coef(fit))
}

{
    phi <- summary(fit)$dispersion
    deviance(fit)/phi + 2 * length(coef(fit))
}

---
body shows deviance(fit)/phi + 2*length(Coef). DEVIANCE form not loglik, so clean qaic won clobber. fixed now

now loading MTL and VAN substrate + pop. then peek structure so we know column names before. building silver

In [ ]:
mtl_substrate <- readRDS("/content/eod_0617/saves_eod/mtl_substrate.rds")
van_substrate <- readRDS("/content/eod_0617/saves_eod/van_substrate.rds")
cma_age_data  <- readRDS("/content/eod_0617/saves_eod/cma_age_data_mtlvan.rds")
str(mtl_substrate, max.level = 1)

List of 5
 $ temp_mat     : num [1:6504, 1:765] 12.3 12.3 12.3 12.3 12.3 ...
  ..- attr(*, "dimnames")=List of 2
 $ truth_factors:Classes ‘data.table’ and 'data.frame':	6504 obs. of  5 variables:
  ..- attr(*, ".internal.selfref")=<pointer: (nil)> 
 $ Z            : num [1:6504, 1:17] 1.553 -0.499 0.904 -0.366 -0.312 ...
  ..- attr(*, "dimnames")=List of 2
 $ da_age       :Classes ‘data.table’ and 'data.frame':	6504 obs. of  7 variables:
  ..- attr(*, ".internal.selfref")=<pointer: (nil)> 
 $ n_da         : int 6504


List of 5
 $ temp_mat     : num [1:6504, 1:765] 12.3 12.3 12.3 12.3 12.3 ...
  ..- attr(*, "dimnames")=List of 2
 $ truth_factors:Classes ‘data.table’ and 'data.frame':	6504 obs. of  5 variables:
  ..- attr(*, ".internal.selfref")=<pointer: (nil)>
 $ Z            : num [1:6504, 1:17] 1.553 -0.499 0.904 -0.366 -0.312 ...
  ..- attr(*, "dimnames")=List of 2
 $ da_age       :Classes ‘data.table’ and 'data.frame':	6504 obs. of  7 variables:
  ..- attr(*, ".internal.selfref")=<pointer: (nil)>
 $ n_da         : int 6504

 ---

 mtl sub. is a list of 5. temp is temp_mat - matrix of 6504 DA x 765 days. not a temp C_column, so silver pulls by indexing the matrix not a column. da_age = 6504 x 7 (pop by age band). turth_factors = latent 3-factor structure. Z = 6504 x 17 modifiers.

 importantly, no n_deaths column anywhere, they're simulated, so b4 diagnosing MTL we need to know if mtl_sim already exists or need be built





check if mtl_sim obj exists before deciding whether chunk 1 reuses it or has to simulate first

In [ ]:
ls(pattern = "sim|mtl|MTL")

[1] "mtl_substrate"   "simulate_counts"

'mtl_substrate''simulate_counts'

---

MTl was never simulated!!!!! so fails to converge implied sim existed and fit broke, there's no sim here.

need to simulate MTL to diagnose. first check sim counts signature so we feed it to right arg - temp_mat, truth_factors, da_age, etc. args(fn) prints param list

In [ ]:
args(simulate_counts)

function (temp_mat, da_long, truth_factors, mmt_da, seed = 42) 
NULL

function (temp_mat, da_long, truth_factors, mmt_da, seed = 42)
NULL

---

simulate_counts(temp_mat, da_long, truth_factors, mmt_da, seed=42). 5 args.

wants da_long not da_age - needs a long format DA table, not wide one in sub. mmt_da = per-DA min-mort-temp vector (chunk 2 builds it as apply(temp_mat, 1, quantile, 0.80)).

where does da_long come from for MTl. find or build from da_age before simulating

check what da_long is and how many DAs.

In [ ]:
str(da_long, max.level = 1)
length(unique(da_long$da_uid))

Classes ‘data.table’ and 'data.frame':	23543640 obs. of  6 variables:
 $ da_idx     : int  1 1 1 1 1 1 1 1 1 1 ...
 $ age_band   : chr  "age_0_64" "age_65_74" "age_75_84" "age_85p" ...
 $ pop        : num  1375 115 75 15 1375 ...
 $ date       : Date, format: "2015-05-01" "2015-05-01" ...
 $ annual_rate: num  0.5 20 50 150 0.5 20 50 150 0.5 20 ...
 $ lambda0    : num  0.00188 0.0063 0.01027 0.00616 0.00188 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)> 


[1] 0

Classes ‘data.table’ and 'data.frame':	23543640 obs. of  6 variables:
 $ da_idx     : int  1 1 1 1 1 1 1 1 1 1 ...
 $ age_band   : chr  "age_0_64" "age_65_74" "age_75_84" "age_85p" ...
 $ pop        : num  1375 115 75 15 1375 ...
 $ date       : Date, format: "2015-05-01" "2015-05-01" ...
 $ annual_rate: num  0.5 20 50 150 0.5 20 50 150 0.5 20 ...
 $ lambda0    : num  0.00188 0.0063 0.01027 0.00616 0.00188 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)>
0

---

da_long = 23.5m rows, 6 cols: da_idx, age_band, pop, date, annual_rate, lambda0. one row per DA x age_band x day

DA id is da_idx (integer), not da_uid. thats why count lmine needs fixing. no CMA column, da_idx starts 1 so cant tell which city from values alone. only unique da_idx count tells us: 7682=toronto, 6504=MTL; 23.5M rows is large so prob one full city. checking next

get count da_idx

In [ ]:
length(unique(da_long$da_idx))
range(da_long$date)

[1] 7694

[1] "2015-05-01" "2019-09-30"

7694
2015-05-012019-09-30

---

7694 unique da_idx = toronto (fermi was ~7682, toronto DA count, not MTL's 6504). dates 2015-05-01 to 2019-09-30 = warm season x 5yr, so this is da_long toronto, leftover from bronze. WRONG CITY!!

build MTL da_long. first check if builder fx exists (make_da_long or similar) - reuse beats hand-rolling. if none = construct from substrate by hand

In [ ]:
ls(pattern = "long|make|build")

[1] "build_crossbasis" "da_long"          "make_choropleth"  "make_strata_A"   
[5] "make_strata_B"    "pop_long"

'build_crossbasis''da_long''make_choropleth''make_strata_A''make_strata_B''pop_long'
---

no make da_long. - no long-table builder exists. so da_long was built inline last session, we reconstruct by hand. have:make_strata_A/B, build_crossbasis (locked, reuse later), pop_long (maybe already long pop - check). pieces for MTL da_long come from substrate da_age + dates + rates

peek pop_long and how lambda0 was derived in torontos da_long - rebuild MTl in same shape not guess formula. lambda0 looks like a daily baseline rate, want to confirm how its computed from annual_rate and pop

In [ ]:
str(pop_long, max.level = 1)
head(da_long[da_long$da_idx == 1, ], 8)

Classes ‘data.table’ and 'data.frame':	30776 obs. of  3 variables:
 $ da_idx  : int  1 2 3 4 5 6 7 8 9 10 ...
 $ age_band: chr  "age_0_64" "age_0_64" "age_0_64" "age_0_64" ...
 $ pop     : num  1375 550 410 340 400 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)> 


da_idx,age_band,pop,date,annual_rate,lambda0
<int>,<chr>,<dbl>,<date>,<dbl>,<dbl>
1,age_0_64,1375,2015-05-01,0.5,0.001883562
1,age_65_74,115,2015-05-01,20.0,0.006301370
1,age_75_84,75,2015-05-01,50.0,0.010273973
1,age_85p,15,2015-05-01,150.0,0.006164384
1,age_0_64,1375,2015-05-02,0.5,0.001883562
1,age_65_74,115,2015-05-02,20.0,0.006301370
1,age_75_84,75,2015-05-02,50.0,0.010273973
1,age_85p,15,2015-05-02,150.0,0.006164384


Classes ‘data.table’ and 'data.frame':	30776 obs. of  3 variables:
 $ da_idx  : int  1 2 3 4 5 6 7 8 9 10 ...
 $ age_band: chr  "age_0_64" "age_0_64" "age_0_64" "age_0_64" ...
 $ pop     : num  1375 550 410 340 400 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)>
A data.table: 8 × 6
da_idx	age_band	pop	date	annual_rate	lambda0
<int>	<chr>	<dbl>	<date>	<dbl>	<dbl>
1	age_0_64	1375	2015-05-01	0.5	0.001883562
1	age_65_74	115	2015-05-01	20.0	0.006301370
1	age_75_84	75	2015-05-01	50.0	0.010273973
1	age_85p	15	2015-05-01	150.0	0.006164384
1	age_0_64	1375	2015-05-02	0.5	0.001883562
1	age_65_74	115	2015-05-02	20.0	0.006301370
1	age_75_84	75	2015-05-02	50.0	0.010273973
1	age_85p	15	2015-05-02	150.0	0.006164384

---

recovered formula exactly: lambda0 = annual+rate * pop / 1000 / 365. annual_rate = deaths per 1000/yr per band (0.5/20/50/150 for the 4 bands), lambda0 = expected daily deaths for that DA x band. Check DA1: 0.5*1375/1000/365=0.001884 match. 85+:150*15/1000/365=0.006164 match

pop_long = 30776 rows = 7964 DAs x 4 bands, torontos. for mtl build same long pop from substrate da_age, cross with dates, apply same formula.

build MTL da_long. need MTLs pop vby band in long form from substrate da_age (6504 x 7 wide). check da_age columns first - which are the 4 pop-by-band cols - before melting to long

In [ ]:
str(mtl_substrate$da_age)
head(mtl_substrate$da_age, 3)

Classes ‘data.table’ and 'data.frame':	6504 obs. of  7 variables:
 $ ALT_GEO_CODE: chr  "24520101" "24520102" "24520103" "24520104" ...
 $ age_0_64    : num  705 505 390 745 470 370 745 655 510 1050 ...
 $ age_65_74   : num  115 50 35 60 50 65 90 115 85 110 ...
 $ age_75_84   : num  45 15 5 15 25 40 45 70 60 55 ...
 $ age_85p     : num  5 5 0 5 0 15 10 25 35 5 ...
 $ total       : num  870 575 430 825 545 490 890 865 690 1220 ...
 $ da_idx      : int  1 2 3 4 5 6 7 8 9 10 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)> 


ALT_GEO_CODE,age_0_64,age_65_74,age_75_84,age_85p,total,da_idx
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
24520101,705,115,45,5,870,1
24520102,505,50,15,5,575,2
24520103,390,35,5,0,430,3


Classes ‘data.table’ and 'data.frame':	6504 obs. of  7 variables:
 $ ALT_GEO_CODE: chr  "24520101" "24520102" "24520103" "24520104" ...
 $ age_0_64    : num  705 505 390 745 470 370 745 655 510 1050 ...
 $ age_65_74   : num  115 50 35 60 50 65 90 115 85 110 ...
 $ age_75_84   : num  45 15 5 15 25 40 45 70 60 55 ...
 $ age_85p     : num  5 5 0 5 0 15 10 25 35 5 ...
 $ total       : num  870 575 430 825 545 490 890 865 690 1220 ...
 $ da_idx      : int  1 2 3 4 5 6 7 8 9 10 ...
 - attr(*, ".internal.selfref")=<pointer: (nil)>
A data.table: 3 × 7
ALT_GEO_CODE	age_0_64	age_65_74	age_75_84	age_85p	total	da_idx
<chr>	<dbl>	<dbl>	<dbl>	<dbl>	<dbl>	<int>
24520101	705	115	45	5	870	1
24520102	505	50	15	5	575	2
24520103	390	35	5	0	430	3

---

da_age = 6504 x 7: ALT_GEO_CODE (Real statcan DA id string), 4 pop cols age_0_64_65_74/75_84/85p, total, da_idx (int key 1..6504). band names match torontos da_long exactly so melt maps clean. annual_rate by band = 0.5/20/50/150 in that order.

built NTL da_long: melt da_age 4 pop cols to long, attach annual_rate by band, cross with 765 date vector, compute lambda0 = annual_Rate*pop/1000/365, assert final row count= 6504 DAs x 4 bands x 765 days before trusting it.

In [ ]:
str(study_dates)
length(study_dates)
ncol(mtl_substrate$temp_mat)

 Date[1:765], format: "2015-05-01" "2015-05-02" "2015-05-03" "2015-05-04" "2015-05-05" ...


[1] 765

[1] 765

 Date[1:765], format: "2015-05-01" "2015-05-02" "2015-05-03" "2015-05-04" "2015-05-05" ...
765
765

---

study_dates = Date[1:765], 2015-05-01 onward. length 765 = ncol(temp_mat) 765. all three agree, cross is safe.

build MTL da_long. melt 4 pop cols long, map annual_rate by band, cross with 765 dates, lambda0 = annual_rate*pop/1000/365. assert nrow = 6504*4*765 = 190902240 before trusting.j

In [ ]:
rate_map <- c(age_0_64 = 0.5, age_65_74 = 20, age_75_84 = 50, age_85p = 150)

mtl_pop_long <- melt(
  mtl_substrate$da_age,
  id.vars = "da_idx",
  measure.vars = names(rate_map),
  variable.name = "age_band",
  value.name = "pop"
)
mtl_pop_long[, age_band := as.character(age_band)]
mtl_pop_long[, annual_rate := rate_map[age_band]]

mtl_da_long <- mtl_pop_long[
  CJ(da_idx = da_idx, date = study_dates, unique = TRUE),
  on = "da_idx", allow.cartesian = TRUE
]
mtl_da_long[, lambda0 := annual_rate * pop / 1000 / 365]

nrow(mtl_da_long)
stopifnot(nrow(mtl_da_long) == 6504L * 4L * 765L)
head(mtl_da_long, 4)

[1] 19902240

da_idx,age_band,pop,annual_rate,date,lambda0
<int>,<chr>,<dbl>,<dbl>,<date>,<dbl>
1,age_0_64,705,0.5,2015-05-01,0.0009657534
1,age_65_74,115,20.0,2015-05-01,0.0063013699
1,age_75_84,45,50.0,2015-05-01,0.0061643836
1,age_85p,5,150.0,2015-05-01,0.0020547945


19902240
A data.table: 4 × 6
da_idx	age_band	pop	annual_rate	date	lambda0
<int>	<chr>	<dbl>	<dbl>	<date>	<dbl>
1	age_0_64	705	0.5	2015-05-01	0.0009657534
1	age_65_74	115	20.0	2015-05-01	0.0063013699
1	age_75_84	45	50.0	2015-05-01	0.0061643836
1	age_85p	5	150.0	2015-05-01	0.0020547945

---

nrow = 19902240 = 6504*4*765. stopifnot silent = cross held, no rows dropped or duped. lambda0 tiny (~0.001-0.006) - correct, it's per person per day rate. mtl da1 pop=705 vs torontos 1375 same band - confirms this is MTL da_age not toronto leaking.

last piece before sim: mmt_da = per-da-min-mort-temp = 80th pct of each DAs temp row. apply(temp_mat, 1, quantile, 0.80). should be length 6504, one per DA in realisitic temp range

In [ ]:
mmt_da <- apply(mtl_substrate$temp_mat, 1, quantile, 0.80)
length(mmt_da)
range(mmt_da)

[1] 6504

[1] 20.53704 22.53167

6504
20.537039097444122.5316664505005

---

mmt_da length 6504 = one MMT per Da. range 20.5-22.5C - 80th pct of each DAs warm-season temps, sits upper-middle as expected, tight because DAs in one metro cluster. sane. all 4 sim inputs now in hand: temp_mat, da_long, truth_Factor, mmt_da

draw MTL 75-84 silver exactly as bronze will: set.seed(42), sample 150 da_idx from all 6504. subset temp_mat, da_long, truth_factors, mmt_da to those same 150 in the same order. assert all four align on da_idx before sim.

In [ ]:
rownames(mtl_substrate$temp_mat)[1:5]
nrow(mtl_substrate$temp_mat)

[1] "24520101" "24520102" "24520103" "24520104" "24520105"

[1] 6504

'24520101''24520102''24520103''24520104''24520105'
6504

---

temp_mat rownames = ALT_GEO_CODE (string), da_long keyed by da_idx (int). bridge is da_Age which carries both cols. must assert da_age ALT_GEO_Code order == temp_mat rownames before simulating - else temps pair with wrong DAs silently

temp_mat keyed by ALT_GEO_CODE, da_long by da_idx - bridge is da_age. assert da_age$ALT_GEO_CODE order == temp_mat rownames and da_age$da_idx == 1:6504. both must hold or sim pairs wrong temp with wrong DA silently.

In [ ]:
stopifnot(
  all(mtl_substrate$da_age$ALT_GEO_CODE == rownames(mtl_substrate$temp_mat)),
  all(mtl_substrate$da_age$da_idx == 1:6504)
)
cat("alignment holds: da_age <-> temp_mat row order coherent, da_idx is 1:6504\n")

alignment holds: da_age <-> temp_mat row order coherent, da_idx is 1:6504


alignment holds: da_age <-> temp_mat row order coherent, da_idx is 1:6504

---

cat printed, both stopifnot passed. da_age ALT_GEO_CODE order == temp_mat rownames, da_idx == 1:6504. then two keys (string code, integer idx) are coherent - temps pair with correct pops!

the moment: simulate full MTL city. all 4 inputs aligned. simulate_Counts(temp_mat, da_long, truth_factors, mmt_da, seed=42) -> mtl_sim with n_deaths. ~19.9M rows so not instant, then immediately check: any NA deaths, range of n_deaths, total deaths vs the ~1057 per silver fermi scaled to full city.

In [ ]:
mtl_sim <- simulate_counts(
  temp_mat      = mtl_substrate$temp_mat,
  da_long       = mtl_da_long,
  truth_factors = mtl_substrate$truth_factors,
  mmt_da        = mmt_da,
  seed          = 42
)

cat("rows:", nrow(mtl_sim), "\n")
cat("NA deaths:", sum(is.na(mtl_sim$n_deaths)), "\n")
cat("range n_deaths:", paste(range(mtl_sim$n_deaths), collapse = " - "), "\n")
cat("total deaths:", sum(mtl_sim$n_deaths), "\n")

Simulating 6504 DAs × 765 days × 4 age bands
Total deaths simulated: 142934 
Mean deaths per DA-day-age: 0.0072 
% zero days: 99.3 %
rows: 19902240 
NA deaths: 0 
range n_deaths: 0 - 36 
total deaths: 142934 


Simulating 6504 DAs × 765 days × 4 age bands
Total deaths simulated: 142934
Mean deaths per DA-day-age: 0.0072
% zero days: 99.3 %
rows: 19902240
NA deaths: 0
range n_deaths: 0 - 36
total deaths: 142934

---

sim ran clean. rows 19902240 match da_long. NA deaths 0 - no missingness. range 0-36 = deaths per DA x band x DAY (not per DA over the period); 99.3% of cells zero, mean 0.0072 - healthy poisson count with long right tail, exactly what mortality data looks like. max 36 in bounds (300 would flag a lambda0/mmt bug, 36 doesn't). total 142934 = plausible 5yr warm-season metro MTL mortality, matches fermi scaled to full city.

so mtl simulates fine. convergence failure is not in simulation, it lives downstream in stage 1.

attempt real stage 1 on MTL 75-84, read failure directly or watch it suceed - maybe stale fx bug was the whole problem and the clobber fixed it. check fit_stage 1 signature first so we assemble its inputs right

In [ ]:
args(fit_stage1)
args(build_crossbasis)

function (cma_age_data, cb_template, cma_label, age_label, verbose = TRUE) 
NULL

function (T_series, lag_max = 21) 
NULL

fit_stage1(cma_age_data, cb_template, cma_label, age_label, verbose=TRUE) - takes the full cma_age data plus labels, subsets internally. don't pre-slice, hand it the whole thing + cma/age labels.
build_cb(t_series, lag_max=21) - takes a temp series + lag max. 5x5=25 knot.

so fit needs: cma_age_data holding MTL, a cb_template from build_crossbasis, labels "MTL" + "age_75_84".

fit_stage 1 wants cma_age_data with MTL deaths. our fresh mtl_sim isn't it. peek cma_age_delta structure then get mtl_sim into the shape it expects

In [ ]:
str(cma_age_data, max.level = 2)

List of 2
 $ montreal :Classes ‘data.table’ and 'data.frame':	6574 obs. of  6 variables:
  ..$ ALT_GEO_CODE: chr [1:6574] "24520101" "24520102" "24520103" "24520104" ...
  ..$ age_0_64    : num [1:6574] 705 505 390 745 470 370 745 655 510 1050 ...
  ..$ age_65_74   : num [1:6574] 115 50 35 60 50 65 90 115 85 110 ...
  ..$ age_75_84   : num [1:6574] 45 15 5 15 25 40 45 70 60 55 ...
  ..$ age_85p     : num [1:6574] 5 5 0 5 0 15 10 25 35 5 ...
  ..$ total       : num [1:6574] 870 575 430 825 545 490 890 865 690 1220 ...
  ..- attr(*, ".internal.selfref")=<pointer: (nil)> 
 $ vancouver:Classes ‘data.table’ and 'data.frame':	3590 obs. of  6 variables:
  ..$ ALT_GEO_CODE: chr [1:3590] "59150004" "59150005" "59150006" "59150007" ...
  ..$ age_0_64    : num [1:3590] 240 435 355 365 430 315 640 395 235 825 ...
  ..$ age_65_74   : num [1:3590] 80 45 55 100 90 45 120 70 30 180 ...
  ..$ age_75_84   : num [1:3590] 40 35 20 60 40 15 75 35 20 130 ...
  ..$ age_85p     : num [1:3590] 10 5 15 10 15 10

List of 2
 $ montreal :Classes ‘data.table’ and 'data.frame':	6574 obs. of  6 variables:
  ..$ ALT_GEO_CODE: chr [1:6574] "24520101" "24520102" "24520103" "24520104" ...
  ..$ age_0_64    : num [1:6574] 705 505 390 745 470 370 745 655 510 1050 ...
  ..$ age_65_74   : num [1:6574] 115 50 35 60 50 65 90 115 85 110 ...
  ..$ age_75_84   : num [1:6574] 45 15 5 15 25 40 45 70 60 55 ...
  ..$ age_85p     : num [1:6574] 5 5 0 5 0 15 10 25 35 5 ...
  ..$ total       : num [1:6574] 870 575 430 825 545 490 890 865 690 1220 ...
  ..- attr(*, ".internal.selfref")=<pointer: (nil)>
 $ vancouver:Classes ‘data.table’ and 'data.frame':	3590 obs. of  6 variables:
  ..$ ALT_GEO_CODE: chr [1:3590] "59150004" "59150005" "59150006" "59150007" ...
  ..$ age_0_64    : num [1:3590] 240 435 355 365 430 315 640 395 235 825 ...
  ..$ age_65_74   : num [1:3590] 80 45 55 100 90 45 120 70 30 180 ...
  ..$ age_75_84   : num [1:3590] 40 35 20 60 40 15 75 35 20 130 ...
  ..$ age_85p     : num [1:3590] 10 5 15 10 15 10 20 15 10 35 ...
  ..$ total       : num [1:3590] 370 520 445 535 575 385 855 515 295 1170 ...
  ..- attr(*, ".internal.selfref")=<pointer: (nil)>  
  
  ---
  
  cma_age_data = list of 2 (montreal, vancouver), each just POP by age band - ALT_GEO_CODE, 4 pop cols, total. No deaths, no temps, no dates. it's a static demographic table.

  two flags: (1) fit_stage1 cant fit off pop alone - either it joins deaths + temp internally or this isnt the right input. (2) montreal here = 6574 rows but mtl_substrate/mtl_sim = 6504 - a 70 da gap.

read fit_stage1 body - signature gave the args, body gives the wiring: how it turns cma_age_Data + cb_template + labels into a gnm fit, where deaths and temp enter, how strata get built.

In [ ]:
print(fit_stage1)

function (cma_age_data, cb_template, cma_label, age_label, verbose = TRUE) 
{
    if (verbose) 
        cat(sprintf("\n=== Stage 1: %s, age %s ===\n", cma_label, 
            age_label))
    cma_age_data[, `:=`(strata_A, make_strata_A(DA_id, date))]
    cma_age_data[, `:=`(strata_B, make_strata_B(DA_id, date))]
    cma_age_data[, `:=`(dow, factor(wday(date)))]
    cma_age_data[, `:=`(t, as.integer(date - min(date)) + 1)]
    results <- list()
    for (variant in c("A", "B")) {
        strata_col <- if (variant == "A") 
            "strata_A"
        else "strata_B"
        formula <- if (variant == "A") {
            n_deaths ~ cb + dow + ns(t, df = 20)
        }
        else {
            n_deaths ~ cb + ns(t, df = 20)
        }
        if (verbose) 
            cat(sprintf("  Variant %s: %d strata, %d rows\n", 
                variant, length(unique(cma_age_data[[strata_col]])), 
                nrow(cma_age_data)))
        cma_age_data[, `:=`(strata_use, get(strata_col))]
        fi

function (cma_age_data, cb_template, cma_label, age_label, verbose = TRUE)
{
    if (verbose)
        cat(sprintf("\n=== Stage 1: %s, age %s ===\n", cma_label,
            age_label))
    cma_age_data[, `:=`(strata_A, make_strata_A(DA_id, date))]
    cma_age_data[, `:=`(strata_B, make_strata_B(DA_id, date))]
    cma_age_data[, `:=`(dow, factor(wday(date)))]
    cma_age_data[, `:=`(t, as.integer(date - min(date)) + 1)]
    results <- list()
    for (variant in c("A", "B")) {
        strata_col <- if (variant == "A")
            "strata_A"
        else "strata_B"
        formula <- if (variant == "A") {
            n_deaths ~ cb + dow + ns(t, df = 20)
        }
        else {
            n_deaths ~ cb + ns(t, df = 20)
        }
        if (verbose)
            cat(sprintf("  Variant %s: %d strata, %d rows\n",
                variant, length(unique(cma_age_data[[strata_col]])),
                nrow(cma_age_data)))
        cma_age_data[, `:=`(strata_use, get(strata_col))]
        fit <- try(gnm(formula, data = cma_age_data, family = quasipoisson(),
            eliminate = strata_use), silent = TRUE)
        if (inherits(fit, "try-error")) {
            if (verbose)
                cat(sprintf("  Variant %s: FAILED to converge\n",
                  variant))
            results[[variant]] <- NULL
            next
        }
        results[[variant]] <- list(fit = fit, qaic = qaic(fit),
            n_strata = length(unique(cma_age_data[[strata_col]])),
            mean_deaths_per_stratum = sum(cma_age_data$n_deaths)/length(unique(cma_age_data[[strata_col]])))
        if (verbose)
            cat(sprintf("  Variant %s: qAIC = %.1f\n", variant,
                results[[variant]]$qaic))
    }
    qaic_A <- if (!is.null(results$A))
        results$A$qaic
    else Inf
    qaic_B <- if (!is.null(results$B))
        results$B$qaic
    else Inf
    winner_var <- if (qaic_A < qaic_B)
        "A"
    else "B"
    if (verbose)
        cat(sprintf("  Winner: variant %s\n", winner_var))
    cb_idx <- grep("^cb", names(coef(results[[winner_var]]$fit)))
    list(cma = cma_label, age = age_label, winner_variant = winner_var,
        coef = coef(results[[winner_var]]$fit)[cb_idx], vcov = vcov(results[[winner_var]]$fit)[cb_idx,
            cb_idx], n_strata_A = if (!is.null(results$A)) results$A$n_strata else NA,
        n_strata_B = if (!is.null(results$B)) results$B$n_strata else NA,
        qaic_A = qaic_A, qaic_B = qaic_B, mean_dps_winner = results[[winner_var]]$mean_deaths_per_stratum,
        cb_template = cb_template)
}
<bytecode: 0x5700c48224a0>

----

fit_stage1 body reveals it wants a long per-da-per-day fitting frame, not the population list. it touches: DA_id, date, n_deaths and a cb column. builds t and dow from date internally. calls gnm(n_deaths ~ cb + dow + ns(t,20), eliminate=strata, family=quasipoisson).

so "Cma_age_data" argument is a lie, it's really the assembled sim frame. we pass mtl_sim (renamed + cb attached) as that arg.

mismatches to fix: da_idx -> DA_id rename; attach cb cross-basis aligned row-for-row to the stacked long table: date + n_deaths already correct

In [ ]:
ls(pattern = "bronze|pipeline|run_stage|fit_cma|stage1")

[1] "fit_stage1"

'fit_stage1'

--

ls shows only fit_stage 1 - no bronzing wrapper. toronto was bronzed inline. we hand-assemble the MTL fitting frame ourselves.

mtl_sim is the long frame fit_stage1 needs - has n_Deaths, date, DA key, 19.9M rows. only missing the cb cross-basis column _ DA_id rename. ~15 min assembly. the 70-DA gap is a non-issue: cma_age_data (6574) is unfiltered demographics, we fit on the substrate (6504, filtered) which is the right object anyway.

assemble fitting frame. cb = build_crossbasis on the temp series in mtl_sim's exact row order. lag needs days consecutive within DA so rows must be DA_major (all days of DA1, then DA2). verify mtl_sim row order before building cb on 19.9M rows - day-major would cross DA boundaries and corrupt the lag silently.

In [ ]:
head(mtl_sim[, .(da_idx, age_band, date, n_deaths)], 10)

da_idx,age_band,date,n_deaths
<int>,<chr>,<date>,<int>
1,age_0_64,2015-05-01,0
1,age_65_74,2015-05-01,0
1,age_75_84,2015-05-01,0
1,age_85p,2015-05-01,0
1,age_0_64,2015-05-02,0
1,age_65_74,2015-05-02,0
1,age_75_84,2015-05-02,0
1,age_85p,2015-05-02,0
1,age_0_64,2015-05-03,0


A data.table: 10 × 4
da_idx	age_band	date	n_deaths
<int>	<chr>	<date>	<int>
1	age_0_64	2015-05-01	0
1	age_65_74	2015-05-01	0
1	age_75_84	2015-05-01	0
1	age_85p	2015-05-01	0
1	age_0_64	2015-05-02	0
1	age_65_74	2015-05-02	0
1	age_75_84	2015-05-02	0
1	age_85p	2015-05-02	0
1	age_0_64	2015-05-03	0
1	age_65_74	2015-05-03	0

---

row order is da_idx -> date -> age_band (band cycles innermost). not pure DA-major - age_Band interleaved at finest level. after subsetting to one band it becomes da_idx across climbing dates = DA-major, correct. but don't trust interleave through a subset: impose order explicitly with setorder (slice, da_idx, date) after band-subset, before building cb.

build + fit MTL 75-84 stage 1. subset mtl_sim to age_75_84, setorder(da_idx,date), get temp (from mtl_sim if present else map from temp_mat), build cb = build_crossbasis(temp, 21), rename da_idx-> DA_id, pass to fit_stage.1 read: does it converge, which varianrt wins, qAIC.

In [ ]:
names(mtl_sim)

[1] "date"        "da_idx"      "age_band"    "pop"         "annual_rate"
 [6] "lambda0"     "date_idx"    "log_rr"      "lambda"      "n_deaths"

'date''da_idx''age_band''pop''annual_rate''lambda0''date_idx''log_rr''lambda''n_deaths'

---

no temp column BUT date_idx is here - the day pointer into temp_mat. so temp = temp_mat[da_idx, date_idx], matrix lookup, no join, can't misalign. log_rr+lambda = the sim's true RR and rate

MTL 75-84 stage 1 fit. subset age_75_84, setorder (da_idx, date), temp via temp_mat[cbind(da_idx,date+idx)] one-shot lookup, cb=build_crossbasis(temp,21), attach cb as matrix-column, rename da_idx->DA_id, fit_stage1. watching: converge? variant winner? qAIC.

In [ ]:
mtl_7584 <- mtl_sim[age_band == "age_75_84"]
setorder(mtl_7584, da_idx, date)

mtl_7584[, temp := mtl_substrate$temp_mat[cbind(da_idx, date_idx)]]
cat("temp range:", paste(round(range(mtl_7584$temp), 1), collapse = " - "), "| NA:", sum(is.na(mtl_7584$temp)), "\n")

cb_mtl <- build_crossbasis(mtl_7584$temp, lag_max = 21)
cat("cb dim:", paste(dim(cb_mtl), collapse = " x "), "\n")

mtl_7584[, cb := cb_mtl]
setnames(mtl_7584, "da_idx", "DA_id")

fit_7584 <- fit_stage1(mtl_7584, cb_template = cb_mtl,
                       cma_label = "MTL", age_label = "age_75_84")

temp range: 0.8 - 29 | NA: 0 
cb dim: 4975560 x 25 


Warning message in `[.data.table`(mtl_7584, , `:=`(cb, cb_mtl)):
“25 column matrix RHS of := will be treated as one vector”


ERROR: Error in `[.data.table`(mtl_7584, , `:=`(cb, cb_mtl)): Supplied 124389000 items to be assigned to 4975560 items of column 'cb'. If you wish to 'recycle' the RHS please use rep() to make this intent clear to readers of your code.


In [ ]:
setnames(mtl_7584, "da_idx", "DA_id")

fit_7584 <- fit_stage1(mtl_7584, cb_template = cb_mtl,
                       cma_label = "MTL", age_label = "age_75_84")

Warning message in set(x, j = name, value = value):
“25 column matrix RHS of := will be treated as one vector”
